In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")

print("🚀 正在加载本地生成的真实 A 股数据张量...")

# 1. 加载我们辛苦清洗出来的神圣张量
X = torch.load("node_features_X.pt")
Y = torch.load("risk_labels_Y.pt")
edge_index = torch.load("graph_edge_index.pt") # 基线模型暂不用边关系，留给下一把高端局

print(f"📊 特征矩阵 X 形状: {X.shape} | 标签矩阵 Y 形状: {Y.shape}")

# 2. 极其致命的工程细节：数据标准化 (Normalization)
# 真实的财务数据都是几十亿、上百亿（比如营业收入），直接喂给神经网络会导致梯度瞬间爆炸 (Loss 变 NaN)
# 我们必须用 Z-Score 标准化，把所有特征压缩到均值为 0，方差为 1 的正态分布里
X_mean = X.mean(dim=0)
X_std = X.std(dim=0) + 1e-5  # 加上一个小常数防止除以 0
X_normalized = (X - X_mean) / X_std

# 3. 定义一个纯前馈神经网络 (MLP Baseline) 作为基准分类器
class BaselineRiskModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super(BaselineRiskModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) # 防止过拟合
        self.fc2 = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        h = self.fc1(x)
        h = self.relu(h)
        h = self.dropout(h)
        out = torch.sigmoid(self.fc2(h))
        return out

input_features = X.shape[1]
model = BaselineRiskModel(input_dim=input_features)

# 4. 定义优化器和损失函数
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
# 因为真实数据中，暴雷的公司（1）远少于健康的公司（0），这叫类别不平衡
# 我们给正样本加上权重 (假设健康公司比暴雷公司多 5 倍)
pos_weight = torch.tensor([5.0]) 
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# 注意：BCEWithLogitsLoss 内部自带了 Sigmoid，所以我们需要给模型输出去掉 Sigmoid
# 为了代码复用，我们在计算 Loss 时直接用最后一层的线性输出
class BaselineRiskModel_Logits(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super(BaselineRiskModel_Logits, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h) # 不加 Sigmoid

model = BaselineRiskModel_Logits(input_dim=input_features)
optimizer = optim.Adam(model.parameters(), lr=0.005)

# 5. 开始疯狂训练！
epochs = 200
print("\n🔥 开始基线模型训练 (Training Baseline)...")

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    # 前向传播与计算损失
    logits = model(X_normalized)
    loss = criterion(logits, Y)
    
    # 反向传播更新权重
    loss.backward()
    optimizer.step()
    
    # 每 20 轮打印一次指标
    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            # 预测概率
            preds_prob = torch.sigmoid(model(X_normalized)).numpy()
            true_y = Y.numpy()
            
            # 转为 0 或 1 标签
            preds_label = (preds_prob > 0.5).astype(float)
            
            # 真实世界业务最看重的不是 Accuracy，而是 AUC！
            acc = accuracy_score(true_y, preds_label)
            try:
                auc = roc_auc_score(true_y, preds_prob)
            except ValueError:
                auc = 0.5 # 防止只有一个类别的报错
            
            print(f"Epoch [{epoch+1:03d}/{epochs}] | Loss: {loss.item():.4f} | Acc: {acc:.2f} | AUC: {auc:.3f}")

print("\n✅ Baseline 测试完毕！")

🚀 正在加载本地生成的真实 A 股数据张量...
📊 特征矩阵 X 形状: torch.Size([597, 5]) | 标签矩阵 Y 形状: torch.Size([597, 1])

🔥 开始基线模型训练 (Training Baseline)...
Epoch [020/200] | Loss: 2.5634 | Acc: 0.70 | AUC: 0.477
Epoch [040/200] | Loss: 2.0160 | Acc: 0.73 | AUC: 0.524
Epoch [060/200] | Loss: 1.4353 | Acc: 0.74 | AUC: 0.579
Epoch [080/200] | Loss: 1.0480 | Acc: 0.74 | AUC: 0.616
Epoch [100/200] | Loss: 0.9425 | Acc: 0.74 | AUC: 0.620
Epoch [120/200] | Loss: 0.9284 | Acc: 0.74 | AUC: 0.634
Epoch [140/200] | Loss: 0.9138 | Acc: 0.74 | AUC: 0.659
Epoch [160/200] | Loss: 0.9022 | Acc: 0.74 | AUC: 0.692
Epoch [180/200] | Loss: 0.8936 | Acc: 0.75 | AUC: 0.712
Epoch [200/200] | Loss: 0.8867 | Acc: 0.75 | AUC: 0.723

✅ Baseline 测试完毕！
